# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Areebaarain/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rank pages higher when they have high search volume and appear stale or have declining search performance. Give higher scores to pages that have a stronger content refresh opportunity, then assign an action based on the score.

In [10]:
!git clone https://github.com/Areebaarain/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# My baseline rule:
# Rank pages higher when they have higher search volume and are more stale.
# The strongest refresh opportunities are pages with both high volume and high staleness.

REASON_CODES = {
    "STALE_HIGH_VOLUME": "Page is stale and has high search volume.",
    "STALE": "Page is stale but has lower search volume.",
    "HIGH_VOLUME": "Page has high search volume but is not especially stale.",
    "LOW_OPPORTUNITY": "Page has neither strong staleness nor high search volume."
}

print("Rule:")
print("Rank pages higher when search volume and staleness indicate a stronger refresh opportunity.")

print("\nReason codes:")
for code, description in REASON_CODES.items():
    print(f"{code}: {description}")



Rule:
Rank pages higher when search volume and staleness indicate a stronger refresh opportunity.

Reason codes:
STALE_HIGH_VOLUME: Page is stale and has high search volume.
STALE: Page is stale but has lower search volume.
HIGH_VOLUME: Page has high search volume but is not especially stale.
LOW_OPPORTUNITY: Page has neither strong staleness nor high search volume.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [12]:

!git clone https://github.com/Areebaarain/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [13]:
import pandas as pd
import os

path = "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

print("File exists:", os.path.exists(path))

df = pd.read_csv(path)

print("Rows:", len(df))
print("Columns:", len(df.columns))

File exists: True
Rows: 30000
Columns: 44


In [14]:
# Print all column names
print("Column names in the dataset:\n")

for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

Column names in the dataset:

1. content_id
2. client_id
3. search_volume
4. competition
5. competition_level
6. cpc
7. content_type
8. main_intent
9. word_count
10. char_count
11. provider_used
12. model_used
13. impressions_90d
14. clicks_90d
15. pageviews_90d
16. sessions_90d
17. users_90d
18. engaged_sessions_90d
19. ai_sessions_90d
20. scroll_events_90d
21. days_with_impressions
22. days_with_sessions
23. impressions_last_30d
24. clicks_last_30d
25. sessions_last_30d
26. impressions_prev_30d
27. clicks_prev_30d
28. sessions_prev_30d
29. content_age_days
30. age_tier
31. age_tier_order
32. days_since_last_update
33. freshness_tier
34. word_count_tier
35. char_count_tier
36. ctr
37. avg_position
38. engagement_rate
39. scroll_rate
40. ai_traffic_pct
41. impression_tier
42. position_tier
43. trend_direction
44. trend_pct


In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd

# Make a copy of the dataset
queue = df.copy()

# -----------------------------
# 1. Calculate baseline score
# -----------------------------
# Higher search volume + more days since update = higher opportunity
queue["score"] = (
    queue["search_volume"].rank(pct=True) * 0.5
    + queue["days_since_last_update"].rank(pct=True) * 0.5
)

# -----------------------------
# 2. Assign reason codes
# -----------------------------
volume_median = queue["search_volume"].median()
stale_median = queue["days_since_last_update"].median()

queue["reason_code"] = "LOW_OPPORTUNITY"

queue.loc[
    (queue["search_volume"] >= volume_median) &
    (queue["days_since_last_update"] >= stale_median),
    "reason_code"
] = "STALE_HIGH_VOLUME"

queue.loc[
    (queue["search_volume"] < volume_median) &
    (queue["days_since_last_update"] >= stale_median),
    "reason_code"
] = "STALE"

queue.loc[
    (queue["search_volume"] >= volume_median) &
    (queue["days_since_last_update"] < stale_median),
    "reason_code"
] = "HIGH_VOLUME"

# -----------------------------
# 3. Assign action labels
# -----------------------------
queue["action"] = "MONITOR"

queue.loc[
    queue["reason_code"] == "STALE_HIGH_VOLUME",
    "action"
] = "REFRESH_NOW"

queue.loc[
    queue["reason_code"] == "STALE",
    "action"
] = "REVIEW_REFRESH"

queue.loc[
    queue["reason_code"] == "HIGH_VOLUME",
    "action"
] = "REVIEW"

# -----------------------------
# 4. Rank everything
# -----------------------------
queue = queue.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

queue["rank"] = queue.index + 1

# -----------------------------
# 5. Write the CSV
# -----------------------------
os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"

queue.to_csv(output_path, index=False)

print(f"Saved {len(queue):,} rows to:")
print(output_path)

# Show top 10
display(
    queue[
        [
            "rank",
            "content_id",
            "search_volume",
            "days_since_last_update",
            "score",
            "reason_code",
            "action"
        ]
    ].head(10)
)



Saved 30,000 rows to:
work/outputs/baseline_action_score.csv


,rank,content_id,search_volume,days_since_last_update,score,reason_code,action
0,1,content_a31e10779c01,3600.0,144,0.992520,STALE_HIGH_VOLUME,REFRESH_NOW
1,2,content_bbca724138f2,1600.0,236,0.991291,STALE_HIGH_VOLUME,REFRESH_NOW
2,3,content_40e140ba2934,720.0,231,0.984434,STALE_HIGH_VOLUME,REFRESH_NOW
3,4,content_24abafed9707,480.0,231,0.978968,STALE_HIGH_VOLUME,REFRESH_NOW
4,5,content_23e958c54c78,480.0,144,0.975976,STALE_HIGH_VOLUME,REFRESH_NOW
5,6,content_c3dd69918c8c,320.0,151,0.969697,STALE_HIGH_VOLUME,REFRESH_NOW
6,7,content_29ec1008c834,320.0,151,0.969697,STALE_HIGH_VOLUME,REFRESH_NOW
7,8,content_6efb8fa48ebe,210.0,151,0.961461,STALE_HIGH_VOLUME,REFRESH_NOW
8,9,content_0cc405838fc5,210.0,144,0.961002,STALE_HIGH_VOLUME,REFRESH_NOW
9,10,content_17e6b2ba4b08,170.0,144,0.956235,STALE_HIGH_VOLUME,REFRESH_NOW


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3 — Top-20 Review

top20 = queue.head(20).copy()

# Confidence note based on the two signals used in the baseline rule
def confidence_note(row):
    if row["reason_code"] == "STALE_HIGH_VOLUME":
        return "High confidence: both search volume and days since last update support a refresh opportunity."
    elif row["reason_code"] == "STALE":
        return "Medium confidence: staleness supports refresh, but search volume is lower."
    elif row["reason_code"] == "HIGH_VOLUME":
        return "Medium confidence: high search volume supports opportunity, but the page is not especially stale."
    else:
        return "Low confidence: neither signal strongly supports a refresh."

def wrong_if(row):
    if row["reason_code"] == "STALE_HIGH_VOLUME":
        return "Wrong if the page is already accurate, still relevant, or the high volume is not valuable traffic."
    elif row["reason_code"] == "STALE":
        return "Wrong if the page remains accurate and relevant despite being old."
    elif row["reason_code"] == "HIGH_VOLUME":
        return "Wrong if the page does not need updating and high volume does not represent a useful opportunity."
    else:
        return "Wrong if another important signal shows a stronger refresh opportunity."

top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(wrong_if, axis=1)

# Display the required review
display(
    top20[
        [
            "rank",
            "content_id",
            "action",
            "reason_code",
            "confidence_note",
            "what_would_make_it_wrong"
        ]
    ]
)


,rank,content_id,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_a31e10779c01,REFRESH_NOW,STALE_HIGH_VOLUME,High confidence: both search volume and days s...,"Wrong if the page is already accurate, still r..."
1,2,content_bbca724138f2,REFRESH_NOW,STALE_HIGH_VOLUME,High confidence: both search volume and days s...,"Wrong if the page is already accurate, still r..."
2,3,content_40e140ba2934,REFRESH_NOW,STALE_HIGH_VOLUME,High confidence: both search volume and days s...,"Wrong if the page is already accurate, still r..."
3,4,content_24abafed9707,REFRESH_NOW,STALE_HIGH_VOLUME,High confidence: both search volume and days s...,"Wrong if the page is already accurate, still r..."
4,5,content_23e958c54c78,REFRESH_NOW,STALE_HIGH_VOLUME,High confidence: both search volume and days s...,"Wrong if the page is already accurate, still r..."
5,6,content_c3dd69918c8c,REFRESH_NOW,STALE_HIGH_VOLUME,High confidence: both search volume and days s...,"Wrong if the page is already accurate, still r..."
6,7,content_29ec1008c834,REFRESH_NOW,STALE_HIGH_VOLUME,High confidence: both search volume and days s...,"Wrong if the page is already accurate, still r..."
7,8,content_6efb8fa48ebe,REFRESH_NOW,STALE_HIGH_VOLUME,High confidence: both search volume and days s...,"Wrong if the page is already accurate, still r..."
8,9,content_0cc405838fc5,REFRESH_NOW,STALE_HIGH_VOLUME,High confidence: both search volume and days s...,"Wrong if the page is already accurate, still r..."
9,10,content_17e6b2ba4b08,REFRESH_NOW,STALE_HIGH_VOLUME,High confidence: both search volume and days s...,"Wrong if the page is already accurate, still r..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4 — Weak picks + leakage check

# Weak picks: lowest-scoring rows from the ranked queue
weak_picks = queue.tail(10)

print("WEAK PICKS")
display(
    weak_picks[
        [
            "rank",
            "content_id",
            "search_volume",
            "days_since_last_update",
            "score",
            "reason_code",
            "action"
        ]
    ]
)

print("\nWHY THESE LOOK WEAK")
print(
    "These picks have lower baseline scores because they do not have "
    "both high search volume and high days since last update. "
    "They are weaker refresh opportunities under this rule."
)

print("\nLEAKAGE CHECK")

# Product/flag-related columns that must NOT be used by the baseline
product_flag_columns = [
    col for col in df.columns
    if "flag" in col.lower() or "product" in col.lower()
]

print("Product/flag columns found in dataset:", product_flag_columns)

# Columns actually used by our baseline
baseline_inputs = [
    "search_volume",
    "days_since_last_update"
]

print("Baseline inputs:", baseline_inputs)

print("\nCONFIRMATION:")
print("✓ No product flags were used in the score or reason code.")
print("✓ No future-window or label-derived inputs were used.")
print("✓ The baseline uses only search_volume and days_since_last_update.")


WEAK PICKS


,rank,content_id,search_volume,days_since_last_update,score,reason_code,action
29990,29991,content_1a6977ff1ef1,NaN,211,NaN,LOW_OPPORTUNITY,MONITOR
29991,29992,content_ba7082c9436c,NaN,6,NaN,LOW_OPPORTUNITY,MONITOR
29992,29993,content_d10c7372129c,NaN,104,NaN,LOW_OPPORTUNITY,MONITOR
29993,29994,content_1d4f4025c930,NaN,8,NaN,LOW_OPPORTUNITY,MONITOR
29994,29995,content_07ce98c6085a,NaN,304,NaN,LOW_OPPORTUNITY,MONITOR
29995,29996,content_0934cd438dd6,NaN,8,NaN,LOW_OPPORTUNITY,MONITOR
29996,29997,content_a3af3b8346d8,NaN,6,NaN,LOW_OPPORTUNITY,MONITOR
29997,29998,content_92a5d2709aa9,NaN,8,NaN,LOW_OPPORTUNITY,MONITOR
29998,29999,content_4be930227848,NaN,20,NaN,LOW_OPPORTUNITY,MONITOR
29999,30000,content_2dfd17269502,NaN,20,NaN,LOW_OPPORTUNITY,MONITOR



WHY THESE LOOK WEAK
These picks have lower baseline scores because they do not have both high search volume and high days since last update. They are weaker refresh opportunities under this rule.

LEAKAGE CHECK
Product/flag columns found in dataset: []
Baseline inputs: ['search_volume', 'days_since_last_update']

CONFIRMATION:
✓ No product flags were used in the score or reason code.
✓ No future-window or label-derived inputs were used.
✓ The baseline uses only search_volume and days_since_last_update.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.